In [1]:
import sqlite3
import pandas as pd

# connect to your database
conn = sqlite3.connect("gtm_funnel_intelligence.db")

query = """
SELECT
    lead_status,
    COUNT(*) AS lead_count,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM leads),
        2
    ) AS percent_of_total
FROM leads
GROUP BY lead_status
ORDER BY lead_count DESC;
"""

df = pd.read_sql_query(query, conn)
df

,lead_status,lead_count,percent_of_total
0,New,1540,30.80
1,MQL,1329,26.58
2,SQL,910,18.20
3,Nurture,737,14.74
4,Disqualified,484,9.68


In [3]:
query = """
WITH funnel AS (
    SELECT
        COUNT(*) AS total_leads,
        SUM(CASE WHEN lead_status = 'MQL' THEN 1 ELSE 0 END) AS mqls,
        SUM(CASE WHEN lead_status = 'SQL' THEN 1 ELSE 0 END) AS sqls
    FROM leads
)

SELECT
    total_leads,
    mqls,
    sqls,
    ROUND(100.0 * mqls / total_leads, 2) AS lead_to_mql_rate,
    ROUND(100.0 * sqls / mqls, 2) AS mql_to_sql_rate,
    ROUND(100.0 * sqls / total_leads, 2) AS lead_to_sql_rate
FROM funnel;
"""

df = pd.read_sql_query(query, conn)
df

,total_leads,mqls,sqls,lead_to_mql_rate,mql_to_sql_rate,lead_to_sql_rate
0,5000,1329,910,26.58,68.47,18.2


In [7]:
query = """
WITH last_activity AS (
    SELECT
        lead_id,
        MAX(activity_date) AS last_activity_date
    FROM activities
    GROUP BY lead_id
),

risk AS (
    SELECT
        o.opportunity_id,
        o.amount,
        o.stage,
        la.last_activity_date,
        CAST(julianday('now') - julianday(la.last_activity_date) AS INTEGER) AS days_since_last_activity,
        CASE
            WHEN CAST(julianday('now') - julianday(la.last_activity_date) AS INTEGER) > 30 THEN 'High Risk'
            WHEN CAST(julianday('now') - julianday(la.last_activity_date) AS INTEGER) > 14 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS pipeline_risk_category
    FROM opportunities o
    LEFT JOIN last_activity la
        ON o.lead_id = la.lead_id
)

SELECT
    pipeline_risk_category,
    SUM(amount) AS total_pipeline
FROM risk
GROUP BY pipeline_risk_category
ORDER BY total_pipeline DESC;
"""

df = pd.read_sql_query(query, conn)
df

,pipeline_risk_category,total_pipeline
0,High Risk,142819710
1,Medium Risk,7172104
2,Low Risk,3242601


In [9]:
query = """
SELECT
    sr.rep_name,
    COUNT(o.opportunity_id) AS total_opportunities,
    SUM(CASE WHEN o.stage = 'Closed Won' THEN o.amount ELSE 0 END) AS closed_won_revenue,
    ROUND(AVG(o.amount), 2) AS avg_deal_size
FROM opportunities o
JOIN sales_reps sr
    ON o.assigned_rep_id = sr.rep_id
GROUP BY sr.rep_name
ORDER BY closed_won_revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df

,rep_name,total_opportunities,closed_won_revenue,avg_deal_size
0,Sales Rep 11,98,3006979,128546.94
1,Sales Rep 9,114,2766591,137480.13
2,Sales Rep 5,111,2742629,124553.96
3,Sales Rep 6,119,2678983,123261.97
4,Sales Rep 7,87,2487116,131392.70
5,Sales Rep 1,118,2125519,127104.10
6,Sales Rep 3,104,1964876,133739.17
7,Sales Rep 8,87,1946858,125238.82
8,Sales Rep 2,98,1527008,117297.49
9,Sales Rep 12,95,1441966,123367.78


In [11]:
query = """
SELECT
    l.lead_source,
    COUNT(o.opportunity_id) AS total_opportunities,
    SUM(CASE WHEN o.stage = 'Closed Won' THEN o.amount ELSE 0 END) AS closed_won_revenue,
    ROUND(
        100.0 * SUM(CASE WHEN o.stage = 'Closed Won' THEN 1 ELSE 0 END)
        / COUNT(o.opportunity_id),
        2
    ) AS win_rate
FROM opportunities o
JOIN leads l
    ON o.lead_id = l.lead_id
GROUP BY l.lead_source
ORDER BY closed_won_revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df

,lead_source,total_opportunities,closed_won_revenue,win_rate
0,Organic Search,144,3971866,22.22
1,Event,167,3737471,18.56
2,LinkedIn,150,3643838,20.00
3,Referral,165,3484129,15.15
4,Paid Search,133,3272278,17.29
5,Email Campaign,161,3136101,14.91
6,Webinar,153,1848535,10.46
7,Partner,127,1786485,8.66


In [13]:
query = """
WITH last_activity AS (
    SELECT
        lead_id,
        MAX(activity_date) AS last_activity_date
    FROM activities
    GROUP BY lead_id
)

SELECT
    o.opportunity_id,
    o.stage,
    o.amount,
    o.expected_close_date,
    sr.rep_name,
    l.lead_source,
    la.last_activity_date,
    CAST(julianday('now') - julianday(la.last_activity_date) AS INTEGER) AS days_since_last_activity
FROM opportunities o
JOIN leads l
    ON o.lead_id = l.lead_id
JOIN sales_reps sr
    ON o.assigned_rep_id = sr.rep_id
LEFT JOIN last_activity la
    ON o.lead_id = la.lead_id
WHERE o.stage NOT IN ('Closed Won', 'Closed Lost')
  AND CAST(julianday('now') - julianday(la.last_activity_date) AS INTEGER) > 30
ORDER BY days_since_last_activity DESC;
"""

df = pd.read_sql_query(query, conn)
df.head(20)

,opportunity_id,stage,amount,expected_close_date,rep_name,lead_source,last_activity_date,days_since_last_activity
0,OPP00714,Negotiation,22672,2025-04-22,Sales Rep 5,Event,2025-01-19,462
1,OPP00821,Negotiation,156090,2025-07-28,Sales Rep 6,Referral,2025-01-24,457
2,OPP00599,Negotiation,12441,2026-05-14,Sales Rep 8,Event,2025-02-13,437
3,OPP00350,Discovery,15217,2026-06-12,Sales Rep 1,Email Campaign,2025-02-17,433
4,OPP00314,Negotiation,188075,2026-01-01,Sales Rep 3,Email Campaign,2025-03-05,417
5,OPP01063,Demo,45910,2026-05-19,Sales Rep 10,Event,2025-03-08,414
6,OPP00365,Negotiation,207741,2025-05-29,Sales Rep 1,Referral,2025-03-10,412
7,OPP00781,Discovery,124357,2026-01-22,Sales Rep 11,Event,2025-03-11,411
8,OPP00253,Demo,168331,2025-06-18,Sales Rep 2,Paid Search,2025-03-12,410
9,OPP00478,Demo,151153,2025-12-03,Sales Rep 11,Webinar,2025-03-12,410


In [15]:
query = """
SELECT
    l.lead_source,
    COUNT(o.opportunity_id) AS total_opps,
    SUM(o.amount) AS pipeline_value,
    SUM(CASE WHEN o.stage = 'Closed Won' THEN o.amount ELSE 0 END) AS closed_won_revenue,
    ROUND(
        100.0 * SUM(CASE WHEN o.stage = 'Closed Won' THEN 1 ELSE 0 END)
        / COUNT(o.opportunity_id),
        2
    ) AS win_rate
FROM opportunities o
JOIN leads l ON o.lead_id = l.lead_id
GROUP BY l.lead_source
ORDER BY closed_won_revenue DESC;
"""
df = pd.read_sql_query(query, conn)
df

,lead_source,total_opps,pipeline_value,closed_won_revenue,win_rate
0,Organic Search,144,17809801,3971866,22.22
1,Event,167,19680518,3737471,18.56
2,LinkedIn,150,19379340,3643838,20.00
3,Referral,165,22744314,3484129,15.15
4,Paid Search,133,17302535,3272278,17.29
5,Email Campaign,161,20349250,3136101,14.91
6,Webinar,153,19498975,1848535,10.46
7,Partner,127,16469682,1786485,8.66


In [ ]:
## Revenue by Sales Rep

This analysis evaluates sales efficiency by comparing pipeline volume, closed revenue, and win rate.

In [17]:
query = """
SELECT
    sr.rep_name,
    COUNT(o.opportunity_id) AS total_opps,
    SUM(o.amount) AS pipeline,
    SUM(CASE WHEN o.stage = 'Closed Won' THEN o.amount ELSE 0 END) AS closed_revenue,
    ROUND(
        100.0 * SUM(CASE WHEN o.stage = 'Closed Won' THEN 1 ELSE 0 END)
        / COUNT(o.opportunity_id),
        2
    ) AS win_rate
FROM opportunities o
JOIN sales_reps sr ON o.assigned_rep_id = sr.rep_id
GROUP BY sr.rep_name
ORDER BY win_rate DESC;
"""
df = pd.read_sql_query(query, conn)
df

,rep_name,total_opps,pipeline,closed_revenue,win_rate
0,Sales Rep 11,98,12597600,3006979,21.43
1,Sales Rep 7,87,11431165,2487116,18.39
2,Sales Rep 5,111,13825490,2742629,18.02
3,Sales Rep 6,119,14668175,2678983,17.65
4,Sales Rep 8,87,10895777,1946858,16.09
5,Sales Rep 9,114,15672735,2766591,15.79
6,Sales Rep 3,104,13908874,1964876,15.38
7,Sales Rep 2,98,11495154,1527008,15.31
8,Sales Rep 1,118,14998284,2125519,15.25
9,Sales Rep 4,73,9323173,1178124,15.07


In [ ]:
**Insight:** 
Sales Rep 11 demonstrates the highest efficiency, while Sales Rep 10 shows low conversion despite similar pipeline volume.

In [19]:
query = """
WITH last_activity AS (
    SELECT
        lead_id,
        MAX(activity_date) AS last_activity_date
    FROM activities
    GROUP BY lead_id
)

SELECT
    o.stage,
    COUNT(*) AS deals,
    SUM(o.amount) AS total_value,
    AVG(julianday('now') - julianday(la.last_activity_date)) AS avg_days_idle
FROM opportunities o
LEFT JOIN last_activity la ON o.lead_id = la.lead_id
WHERE o.stage NOT IN ('Closed Won', 'Closed Lost')
GROUP BY o.stage
ORDER BY avg_days_idle DESC;
"""
df = pd.read_sql_query(query, conn)
df

,stage,deals,total_value,avg_days_idle
0,Discovery,241,32125885,135.263467
1,Negotiation,181,23178462,130.555442
2,Proposal,216,26476134,129.485689
3,Demo,213,26971811,127.025982


In [ ]:
SQL analysis confirmed that Organic Search leads generate the highest revenue and win rates, which is reflected in the Power BI dashboard